In [ ]:
# Setup root path one folder up
from pathlib import Path
import sys
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
MODEL_PATH = Path.cwd() / "xgbc_model.joblib"

In [ ]:
# Imports
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import helper as h
import joblib

# Prepare Data

In [ ]:
# Load the raw tables
clients = pd.read_csv(ROOT / "data/train/client_train.csv")
invoices = pd.read_csv(ROOT / "data/train/invoice_train.csv", low_memory=False)

In [ ]:
# Stratified train/holdout split at client level
train_clients, holdout_clients = train_test_split(
    clients,
    test_size=0.2,
    stratify=clients["target"],
    random_state=42,
)

train_invoices = invoices[invoices["client_id"].isin(train_clients["client_id"])]
holdout_invoices = invoices[invoices["client_id"].isin(holdout_clients["client_id"])]

# Build features

In [ ]:
# Build the feature tables for train and holdout
train_features = h.build_features(train_clients, train_invoices)
holdout_features = h.build_features(holdout_clients, holdout_invoices)

# Self-check: one row per client, same columns everywhere
for name, feats, clients in [("train", train_features, train_clients),
                             ("holdout", holdout_features, holdout_clients)]:
    assert feats.index.is_unique and len(feats) == len(clients), name
    print(f"{name:8s} {feats.shape}")

In [ ]:
columns_to_drop = ["target"]

X_train_xgbc = train_features.drop(columns=columns_to_drop)
X_holdout_xgbc = holdout_features.drop(columns=columns_to_drop)

y_train = train_features["target"]
y_holdout = holdout_features["target"]

# Training and Export

In [ ]:
from sklearn.model_selection import RandomizedSearchCV,TunedThresholdClassifierCV, StratifiedKFold
from scipy.stats import randint, uniform, loguniform
from sklearn.metrics import make_scorer, fbeta_score
from xgboost import XGBClassifier

# scoring
f2_scorer = make_scorer(fbeta_score, beta=2)
scoring = {'f2': f2_scorer, 'roc_auc': 'roc_auc'}

#shuffling cv
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Model
xgbc_model = XGBClassifier(objective='binary:logistic', random_state=42, n_jobs=1)

# Threshold tuner
tuned = TunedThresholdClassifierCV(
    estimator=xgbc_model,
    scoring=f2_scorer,
    thresholds=100,      
    cv=cv,                
    refit=True,
    random_state=42,
)

# Hyperparameters
scale = (y_train == 0).sum() / (y_train == 1).sum()

params = {
    'estimator__max_depth': randint(3, 9),
    'estimator__min_child_weight': randint(1, 8),
    'estimator__subsample': uniform(0.6, 0.4),
    'estimator__colsample_bytree': uniform(0.6, 0.4),
    'estimator__learning_rate': loguniform(0.01, 0.3),
    'estimator__n_estimators': randint(100, 500),
    'estimator__scale_pos_weight': [1, scale / 2, scale],
}

# Training
rndsearch = RandomizedSearchCV(n_iter=10, estimator=tuned, param_distributions=params, cv=cv, n_jobs=-1, verbose=2, scoring=scoring, refit="f2", random_state=42)
rndsearch.fit(X_train_xgbc, y_train)

best_xgbc_model = rndsearch.best_estimator_

best_idx = rndsearch.best_index_
print(f"Best CV F2 score: {rndsearch.best_score_:.4f}")
print(f"Best CV ROC-AUC: {rndsearch.cv_results_['mean_test_roc_auc'][best_idx]:.4f}"
      f"(+/- {rndsearch.cv_results_['std_test_roc_auc'][best_idx]:.4f})")
print(f"Best threshold: {best_xgbc_model.best_threshold_}")


In [ ]:
# Export trained tuner with model

joblib.dump(best_xgbc_model, MODEL_PATH)

# Metrics and Plots

In [ ]:
# Use import if not interested in training
MODEL_PATH = Path.cwd() / "xgbc_model.joblib"
best_xgbc_model = joblib.load(MODEL_PATH)

In [ ]:
from sklearn.metrics import roc_auc_score, recall_score, precision_score, classification_report, fbeta_score

y_holdout = holdout_features["target"]

train_proba = best_xgbc_model.predict_proba(X_train_xgbc)[:, 1]   # probabilities for ROC-AUC
test_proba = best_xgbc_model.predict_proba(X_holdout_xgbc)[:, 1]   # probabilities for ROC-AUC
y_pred = best_xgbc_model.predict(X_holdout_xgbc)              # using the tuned threshold

print(f"Holdout ROC-AUC:   {roc_auc_score(y_holdout, test_proba):.4f}")
print(f"Holdout F2:        {fbeta_score(y_holdout, y_pred, beta=2):.4f}")
print(f"Holdout recall:    {recall_score(y_holdout, y_pred):.4f}")
print(f"Holdout precision: {precision_score(y_holdout, y_pred):.4f}")
print(classification_report(y_holdout, y_pred))

# Business value: metrics on the top 10% most probable frauds
business_report = h.top_fraction_report(y_holdout, test_proba, frac=0.10)
print("Business value (top 10% most probable frauds):")
for metric, value in business_report.items():
    print(f"  {metric:10s} {value:.4f}" if isinstance(value, float) else f"  {metric:10s} {value}")

# Compare to test
training_report = h.top_fraction_report(y_train, train_proba, frac=0.10)
print("Compared to Test (top 10% most probable frauds):")
for metric, value in training_report.items():
    print(f"  {metric:10s} {value:.4f}" if isinstance(value, float) else f"  {metric:10s} {value}")

In [ ]:
# Compare ROC-AUC, recall and F2 on train vs. holdout (test) data
import numpy as np

def score_split(model, X, y):
    proba = model.predict_proba(X)[:, 1]
    pred = model.predict(X)  # uses the tuned threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "Recall": recall_score(y, pred),
        "F2": fbeta_score(y, pred, beta=2),
    }

scores = pd.DataFrame({
    "Train": score_split(best_xgbc_model, X_train_xgbc, y_train),
    "Test": score_split(best_xgbc_model, X_holdout_xgbc, y_holdout),
})

# Two bars per metric, touching: bar width = half the group slot
x = np.arange(len(scores.index))
width = 0.4
fig, ax = plt.subplots(figsize=(7, 5))
bars_train = ax.bar(x - width / 2, scores["Train"], width, label="Train", color="#4C72B0")
bars_test = ax.bar(x + width / 2, scores["Test"], width, label="Test", color="#DD8452")

for bars in (bars_train, bars_test):
    ax.bar_label(bars, fmt="%.3f", padding=2)

ax.set_xticks(x)
ax.set_xticklabels(scores.index)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("XGBoost: train vs. test performance")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance, most to least important
# The tuner wraps the fitted XGBClassifier in .estimator_
importances = pd.Series(
    best_xgbc_model.estimator_.feature_importances_,
    index=X_train_xgbc.columns,
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 0.4 * len(importances) + 1))
ax.barh(importances.index, importances.values, color="#4C72B0", zorder=3)
ax.invert_yaxis()  # most important feature at the top

# Light grid lines every 0.01 along the importance axis, drawn behind the bars
ax.set_xticks(np.arange(0, importances.max() + 0.01, 0.01), minor=True)
ax.grid(axis="x", which="both", color="lightgray", alpha=0.6, linewidth=0.6, zorder=0)

ax.set_xlabel("Feature importance")
ax.set_title("XGBoost: feature importance")
plt.tight_layout()
plt.show()